# Chapter 15 &mdash; $Halt_{TM}$ is Undecidable, by Reduction from $A_{TM}$

**Concept 6 of the Chapter 15 decomposition:** *$Halt_{TM}$ is Undecidable, by Reduction from $A_{TM}$*

A halting decider would let you safely run $M$ on $w$ &mdash; giving an $A_{TM}$ decider.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-Halt-TM-Undecidable/Concept-Halt-TM-Undecidable.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$Halt_{TM} = \{\langle M,w\rangle : M \text{ halts on } w\}$$

**Suppose** $H$ decides $Halt_{TM}$. Then decide $A_{TM}$ as follows:

> on $\langle M,w\rangle$: run $H$. If $H$ says "does not halt", **reject** &mdash; a
> machine that never halts certainly never accepts. If $H$ says "halts", **simulate
> $M$ on $w$**; the simulation is now guaranteed to terminate, so report what it says.

That is a decider for $A_{TM}$, which does not exist. So $H$ does not exist.

The shape is the **reduction**: $A_{TM} \le Halt_{TM}$. Note what makes it work &mdash;
the halting oracle removes exactly the risk that made simulation a semi-decider.

## 2. Definitions

### The reduction, executable with a stand-in oracle

In [ ]:
# Stand-in machines: (function, halts_on) where halts_on says which inputs
# the machine halts on.  The oracle H is given to us by assumption.
PROGS = {
 'P1': (lambda w: w.startswith('1'), lambda w: True),               # total
 'P2': (lambda w: True,              lambda w: w.startswith('1')),  # partial
 'P3': (lambda w: False,             lambda w: len(w) % 2 == 0),    # partial
}

def H(prog, w):                    # the ASSUMED halting decider
    return PROGS[prog][1](w)

def simulate(prog, w):             # only safe when H says it halts
    assert H(prog, w), "simulating a machine that does not halt!"
    return PROGS[prog][0](w)

### $A_{TM}$ decided, using $H$

In [ ]:
def decide_ATM(prog, w):
    if not H(prog, w):
        return False               # never halts => never accepts
    return simulate(prog, w)       # now safe: the simulation terminates

## 3. Tests

The constructed decider answers on every input.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(1, 4) for p in product('01', repeat=k)]
for prog in sorted(PROGS):
    answers = [decide_ATM(prog, w) for w in strs]
    print("  %-4s decided all %d inputs; accepted %d of them"
          % (prog, len(answers), sum(answers)))
    assert len(answers) == len(strs)

And it answers **correctly**: accept iff the machine halts AND accepts.

In [ ]:
for prog in sorted(PROGS):
    f, h = PROGS[prog]
    for w in strs:
        want = h(w) and f(w)
        assert decide_ATM(prog, w) == want, (prog, w)
print("decide_ATM matches 'halts and accepts' on every case")

**The oracle is doing the work.** Without it, simulation is unsafe.

In [ ]:
try:
    simulate('P2', '0')            # P2 does not halt on '0'
except AssertionError as e:
    print("without the oracle :", e)
print()
print("H is exactly what converts the SEMI-decider 'run M on w' into a decider.")

So the reduction closes.

In [ ]:
print("if Halt_TM were decidable")
print("   then decide_ATM above decides A_TM")
print("   but A_TM is undecidable (Concept 5)")
print("therefore Halt_TM is undecidable.")

The direction matters: reduce the **known-hard** problem **to** the new one.

In [ ]:
print("A_TM  <=  Halt_TM       correct: solving Halt would solve A_TM")
print("Halt_TM  <=  A_TM       also true here, but proves nothing NEW")
print()
print("To show X is hard, reduce a KNOWN hard problem TO X.")
print("Getting this backwards is the most common error in the chapter.")

## 4. Exercises


1. Write the reduction in the wrong direction and say exactly what it fails to prove.
2. Is $Halt_{TM}$ RE? Is its complement?
3. Reduce $A_{TM}$ to "does $M$ ever write a blank?"

In [ ]:
# Your work for the exercises above.